# Model 3A: OPUS-MT Direct Pretrained Evaluation

This notebook does only one thing: load the pretrained `Helsinki-NLP/opus-mt-en-ig` model, save a local copy of it, and evaluate it on the same English-Igbo test sets used for your other models.

In [1]:
# Run once in the active VS Code environment if needed.
# %pip install -q transformers sacrebleu sentencepiece sacremoses pandas

## 1. Imports and Configuration

In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

MODEL_NAME = "Helsinki-NLP/opus-mt-en-ig"
MODEL_LABEL = "direct_pretrained"
GENERATION_MAX_NEW_TOKENS = 80
GENERATION_NUM_BEAMS = 4
MAX_SOURCE_LENGTH = 128
BATCH_SIZE = 16

EVAL_SET_SPECS = [
    {"label": "external_jw_bible", "filename": "Combined_test.jsonl"},
    {"label": "personal_phrasebook", "filename": "RNN_evaluation_set.jsonl"},
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.executable)
print("PyTorch version:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: c:\python312\python.exe
PyTorch version: 2.10.0+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## 2. Project Paths

In [3]:
PROJECT_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path(r"C:\Users\Mr. Paul\Downloads\CS156 Final Assignment"),
]


def resolve_project_dir(candidates):
    for candidate in candidates:
        if (candidate / "Cleaned Data" / "Combined_test.jsonl").exists():
            return candidate
    return None


PROJECT_DIR = resolve_project_dir(PROJECT_DIR_CANDIDATES)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not locate the project directory.")

DATA_DIR = PROJECT_DIR / "Cleaned Data"
OUTPUT_DIR = PROJECT_DIR / "OPUS_MT Outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUTPUT_DIR / "1_direct_pretrained_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_INFO_PATH = OUTPUT_DIR / "1_direct_pretrained_model_info.json"
REGISTRY_PATH = OUTPUT_DIR / "model_registry.json"

eval_set_paths = [
    {**spec, "path": DATA_DIR / spec["filename"]}
    for spec in EVAL_SET_SPECS
]

for spec in eval_set_paths:
    if not spec["path"].exists():
        raise FileNotFoundError(f"Missing evaluation file: {spec['path']}")

print("Project directory:", PROJECT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Saved pretrained model directory:", MODEL_DIR)
for spec in eval_set_paths:
    print(f"Eval set: {spec['label']} -> {spec['path']}")

Project directory: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment
Output directory: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\OPUS_MT Outputs
Saved pretrained model directory: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\OPUS_MT Outputs\1_direct_pretrained_model
Eval set: external_jw_bible -> c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\Cleaned Data\Combined_test.jsonl
Eval set: personal_phrasebook -> c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\Cleaned Data\RNN_evaluation_set.jsonl


## 3. Data Helpers

In [4]:
def load_jsonl_records(path):
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def summarize_records(label, records):
    return {
        "split": label,
        "rows": len(records),
        "avg_english_chars": round(sum(len(row["English"]) for row in records) / len(records), 2),
        "avg_igbo_chars": round(sum(len(row["Igbo"]) for row in records) / len(records), 2),
    }


eval_sets = []
for spec in eval_set_paths:
    records = load_jsonl_records(spec["path"])
    eval_sets.append({**spec, "records": records})

display(pd.DataFrame([summarize_records(spec["label"], spec["records"]) for spec in eval_sets]))

,split,rows,avg_english_chars,avg_igbo_chars
0,external_jw_bible,6390,127.84,125.86
1,personal_phrasebook,250,125.60,122.12


## 4. Load and Save the Pretrained Model

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
# Avoid duplicate length-control warnings during generation.
model.generation_config.max_length = None
model.eval()

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

model_info = {
    "model_label": MODEL_LABEL,
    "model_name": MODEL_NAME,
    "path": str(MODEL_DIR),
    "description": "Original OPUS-MT model with no fine-tuning on the project dataset.",
}
MODEL_INFO_PATH.write_text(json.dumps(model_info, indent=2), encoding="utf-8")

registry = []
if REGISTRY_PATH.exists():
    registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))

registry = [row for row in registry if row.get("model_label") != MODEL_LABEL]
registry.append(model_info)
REGISTRY_PATH.write_text(json.dumps(registry, indent=2), encoding="utf-8")

print("Loaded model:", MODEL_NAME)
print("Saved pretrained model to:", MODEL_DIR)
print("Model info path:", MODEL_INFO_PATH)
print("Model registry path:", REGISTRY_PATH)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded model: Helsinki-NLP/opus-mt-en-ig
Saved pretrained model to: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\OPUS_MT Outputs\1_direct_pretrained_model
Model info path: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\OPUS_MT Outputs\1_direct_pretrained_model_info.json
Model registry path: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\OPUS_MT Outputs\model_registry.json


## 5. Translation and Evaluation Helpers

In [6]:
bleu_metric = BLEU()
chrf_metric = CHRF()


def translate_batch(texts, batch_size=BATCH_SIZE):
    outputs = []
    model.eval()

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LENGTH,
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}

        with torch.no_grad():
            generated = model.generate(
                **encoded,
                max_new_tokens=GENERATION_MAX_NEW_TOKENS,
                num_beams=GENERATION_NUM_BEAMS,
            )

        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))

    return outputs


def evaluate_records(records, label):
    source_texts = [row["English"] for row in records]
    references = [row["Igbo"] for row in records]
    predictions = translate_batch(source_texts)

    refs_for_sacrebleu = [[text] for text in references]
    exact_match = sum(pred == ref for pred, ref in zip(predictions, references)) / len(predictions)
    bleu = bleu_metric.corpus_score(predictions, refs_for_sacrebleu).score
    chrf = chrf_metric.corpus_score(predictions, refs_for_sacrebleu).score

    metrics_df = pd.DataFrame(
        {
            "metric": ["Exact match", "BLEU", "chrF"],
            "value": [round(exact_match, 4), round(bleu, 2), round(chrf, 2)],
        }
    )
    preview_df = pd.DataFrame(
        {
            "English": source_texts,
            "Reference Igbo": references,
            "Predicted Igbo": predictions,
        }
    )

    metrics_path = OUTPUT_DIR / f"opus_mt_direct_pretrained_{label}_metrics.csv"
    preview_path = OUTPUT_DIR / f"opus_mt_direct_pretrained_{label}_preview.csv"
    metrics_df.to_csv(metrics_path, index=False)
    preview_df.to_csv(preview_path, index=False)
    return metrics_df, preview_df, metrics_path, preview_path

## 6. Evaluate the Pretrained Model

In [7]:
summaries = []

for spec in eval_sets:
    metrics_df, preview_df, metrics_path, preview_path = evaluate_records(spec["records"], spec["label"])
    summary = {"split": spec["label"], "model_variant": MODEL_LABEL, "rows": len(spec["records"])}
    summary.update({row["metric"]: row["value"] for _, row in metrics_df.iterrows()})
    summaries.append(summary)

    print("Saved metrics to:", metrics_path)
    print("Saved preview to:", preview_path)
    display(metrics_df)
    display(preview_df.head(10))

display(pd.DataFrame(summaries))

Saved metrics to: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\OPUS_MT Outputs\opus_mt_direct_pretrained_external_jw_bible_metrics.csv
Saved preview to: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\OPUS_MT Outputs\opus_mt_direct_pretrained_external_jw_bible_preview.csv


,metric,value
0,Exact match,0.002
1,BLEU,18.210
2,chrF,37.060


,English,Reference Igbo,Predicted Igbo
0,The LORD God made a woman from the rib which h...,"Mgbe ahụ, Onyenwe anyị Chineke ji otu ọgịrịga ...",ONYE amụma bụ́ Chineke mere nwanyị n'ọ́gụ̀ ahụ...
1,Others jailed includes; Adamu Mohammed who is ...,Ụfọdụ ndị ọzọ atụrụ mkpọrọ bụ Adamu Mohammed d...,Ndị ọzọ na - agụnye ndị a tụrụ mkpọrọ; Adamu a...
2,For who makes you different? And what do you h...,Onye gwara gị na ị pụrụ iche n'etiti mmadụ ndị...,"N'ihi na ònye mere gị ka ị dị iche, gịnịkwa ka..."
3,"you want to buy something, go to these places","Ị chọrọ izu ihe, gaa ebe ndị a","Ị chọrọ ịzụ ihe, gaa n'ebe ndị a"
4,This company has stopped work there and sacked...,Ụlọọrụ a akwụsịla ọrụ n'ebe ahụ ma chụpụ ndị ọ...,Ụlọ ọrụ a akwụsịla ọrụ n'ebe ahụ ma tinye ndị ...
5,"After about one hour passed, another confident...","Mgbe ihe dị ka otu awa gafere, onye ọzọ bịakwa...","Mgbe ihe dị ka otu awa gasịrị, onye ọzọ ji obi..."
6,"He found a fresh jawbone of a donkey, put out ...","Mgbe ahụ Samsin lere anya gburugburu, hụ ọkpụk...","Ọ hụrụ ọkpụkpụ ịnyịnya ibu ọhụrụ, jide ya n'ak..."
7,But he abandoned the counsel of the old men wh...,"Ma ọ nabataghị ndụmọdụ ndị okenye nyere ya, ọ ...",Ma ọ hapụrụ ndụmọdụ nke ndị agadi ahụ ha nyere...
8,The Federal Government could have simply incre...,Gọọmentị etiti gaara enyekwu ikike nye ụlọ ọrụ...,Gọọmenti etiti gaara eme nnọọ ka e nwekwuo ike...
9,"Stop Attacking Shoprite In Nigeria, They're Ow...",Kwụsị ịwakpo oke ụlọahịa na mba Naịjirịa. Ọ bụ...,"N'obodo Nigeria, a na - akpọ ha Ndị Agha Obodo..."


Saved metrics to: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\OPUS_MT Outputs\opus_mt_direct_pretrained_personal_phrasebook_metrics.csv
Saved preview to: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\OPUS_MT Outputs\opus_mt_direct_pretrained_personal_phrasebook_preview.csv


,metric,value
0,Exact match,0.004
1,BLEU,6.270
2,chrF,20.820


,English,Reference Igbo,Predicted Igbo
0,"and Shallum became the father of Jekamiah, and...","Shalum bụ nna Jekamaya, ebe Jekamaya mụrụ Elis...","Belum ghọrọ nna Jkmila, Jkna ghọkwara nna Ịlaị..."
1,"These are the sons of Bilhah, whom Laban gave ...",Ndị a bụ ụmụ ndị ikom a mụụrụ Jekọb site na Bi...,Ụmụ ndị a bụ ụmụ nwanyị iri na abụọ ahụ Leban ...
2,people were commenting on social media like Tw...,"Na soshal midia di ka Twitter, ndi mmadụ na-ek...",Ndị mmadụ nọ na - ekwu banyere usoro mgbasa oz...
3,You were running well! Who interfered with you...,"Unu na-agbabu ọsọ nke ọma, onye gbochiri unu i...",Ònye gwara gị na i kwesịghị irube isi n'eziokw...
4,"It will never be inhabited, neither will it be...",Ọ gaghị abụ ebe obibi ọzọ ruo mgbe ebighị ebi....,"Ọ dịghị mgbe a ga - ebi n'ime ya, ọ dịghịkwa m..."
5,Mr Okereke said that his office was in charge ...,Mazị Okereke kwuru na ụlọ ọrụ ya na-ahụ maka ị...,Onyeisi ndị uwe ojii ahụ kwuru na ụlọ ọrụ ya n...
6,Buhari said that he has not forgotten his prom...,Buhari kwuru na ya echọzọbeghị nkwa ya kwere ị...,Bovae kwuru na ya echefughị nkwa o kwere na a ...
7,The heads of commonwealth are meeting to discu...,Ndị isi Commonwealth na-ezukọ ka ha kparịta ụk...,Ndị isi nke ụlọ ọrụ ndị nkịtị na - ezukọ iji k...
8,"Another person, name withheld, said that their...",Onye ọzọ na ebe ahụ achoghi ka akpọ aha ha sịr...,"Onye ọzọ, bụ́ onye a na - akpọghị aha ya, kwur..."
9,for the weapons of our warfare are not of the ...,"N'ihi na ngwa agha anyị ji ebu agha, abụghị ng...",N'ihi na ngwá agha anyị ji ebu agha abụghị nke...


,split,model_variant,rows,Exact match,BLEU,chrF
0,external_jw_bible,direct_pretrained,6390,0.002,18.21,37.06
1,personal_phrasebook,direct_pretrained,250,0.004,6.27,20.82


## 7. Manual Translation Check

In [8]:
def translate_text(text):
    return translate_batch([text], batch_size=1)[0]


english_text = "How are you?"
predicted_igbo = translate_text(english_text)

print("Model variant:", MODEL_LABEL)
print("English:", english_text)
print("Predicted Igbo:", predicted_igbo)

Model variant: direct_pretrained
English: How are you?
Predicted Igbo: Olee otú i si eme?
